In [1]:
import sys
from pathlib import Path

# Add src/ to Python path for local module imports
sys.path.insert(0, str(Path.cwd().parent / "src"))

In [10]:
import pandas as pd
import matplotlib.pyplot as plt


import seaborn as sns
import random
import json

import constants
from model_utils import *

from ast import literal_eval
from pathlib import Path

import matplotlib.cm as cm

from matplotlib.colors import LinearSegmentedColormap

import numpy as np

from itertools import combinations

In [3]:
#plt.style.use('ggplot')
sns.set_palette('hls', 18)
SPLIT = 'test'
model_type = constants.RectalCancerStagingData
base_dir = Path.cwd().parent

In [4]:
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 9,          # i caption in LaTeX sono spesso 9-10pt
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'figure.dpi': 150,       # per la preview; il salvataggio usa il suo
})

# Total performance

In [5]:
scores = pd.read_csv(base_dir / 'data' / 'metrics' / 'scores.csv')
scores.rename(
    columns={
        constants.OPENAI_GPT_4_1_NANO: 'GPT 4.1 Nano',
        f'{constants.OPENAI_GPT_4_1_NANO}_few-shots': 'GPT 4.1 Nano\nMMR (lambda=1.0)',
        f'{constants.OPENAI_GPT_4_1_NANO}_MMR': 'GPT 4.1 Nano\nMMR (lambda=0.5)',
        constants.TUNED_GPT_4_1_NANO: 'GPT 4.1 Nano FT',
        constants.TUNED_GPT_4_1_NANO_OVERSAMPLE: 'GPT 4.1 Nano FT OS',
        constants.OPENAI_GPT_4_1_MINI: 'GPT 4.1 Mini',
        constants.OPENAI_GPT_4_1: 'GPT 4.1',
        constants.TUNED_GPT_4_1: 'GPT 4.1 FT',
        #constants.TUNED_GPT_4_1_OVERSAMPLING: 'gpt_4_1_tuned_oversampling',
        #f'few_shots_{constants.TUNED_GPT_4_1_OVERSAMPLING}': 'gpt_4_1_tuned_few_shots',
        #f'MMR_{constants.TUNED_GPT_4_1_OVERSAMPLING}': 'gpt_4_1_tuned_MMR',
        constants.OPENAI_GPT_5_4: 'GPT 5.4',
        constants.MISTRAL_LARGE_3: 'Mistral Large 3',
        constants.CLAUDE_OPUS_4_6: 'Opus 4.6',
        #f'few_shots_{constants.CLAUDE_OPUS_4_6}': 'opus_4_6_few_shots',
        #f'MMR_{constants.CLAUDE_OPUS_4_6}': 'opus_4_6_MMR'
        constants.LLAMA_3_2_3B_TUNED: 'Llama 3.2 3B Tuned',
    },
    inplace=True
)

print(scores.split.value_counts())

train_scores = scores[scores.split=='train']
validation_scores = scores[scores.split=='validation']
test_scores = scores[scores.split=='test']

if SPLIT == 'val-test':
    scores = pd.concat([validation_scores, test_scores], ignore_index=True)
elif SPLIT == 'test':
    scores = test_scores
    
scores.set_index('id', inplace=True)

print(len(scores))

split
train         187
test           65
validation     63
Name: count, dtype: int64
65


In [6]:
scores.head()

,split,Opus 4.6,GPT 4.1 FT,GPT 4.1 Nano FT OS,GPT 4.1 Nano FT,GPT 4.1,GPT 4.1 Mini,GPT 4.1 Nano,GPT 4.1 Nano\nMMR (lambda=0.5),GPT 4.1 Nano\nMMR (lambda=1.0),GPT 5.4,Llama 3.2 3B Tuned,Mistral Large 3
id,,,,,,,,,,,,,
46,test,0.769774,0.689266,0.731638,0.745763,0.766949,0.766949,0.738701,0.710452,0.710452,0.795198,0.632768,0.826271
47,test,1.000000,1.000000,0.915254,0.971751,1.000000,1.000000,0.774011,0.802260,0.717514,1.000000,0.858757,1.000000
53,test,0.706215,0.915254,0.774011,0.830508,0.745763,0.742938,0.536723,0.536723,0.536723,0.887006,0.564972,0.745763
54,test,0.731638,0.717514,0.661017,0.756121,0.661017,0.774011,0.706215,0.706215,0.734463,0.717514,0.457627,0.736347
56,test,0.661017,0.830508,0.887006,0.887006,0.830508,0.689266,0.649718,0.649718,0.621469,0.830508,0.819209,0.689266


In [7]:
compared_models = [
    #'GPT 4.1 Nano',
    #'GPT 4.1 Nano\nMMR (lambda=1.0)',
    #'GPT 4.1 Nano\nMMR (lambda=0.5)',
    #'GPT 4.1 Nano FT',
    #'GPT 4.1 Nano FT OS',
    #'Llama 3.2 3B Tuned',
    #'Llama 3.2 1B Tuned',
    #'GPT 4.1 Mini',
    'GPT 4.1',
    'GPT 4.1 FT',
    'GPT 5.4',
    'Mistral Large 3',
    'Opus 4.6'
]
scores = scores[compared_models]

In [11]:
display(scores.head())

,GPT 4.1,GPT 4.1 FT,GPT 5.4,Mistral Large 3,Opus 4.6
id,,,,,
46,0.766949,0.689266,0.795198,0.826271,0.769774
47,1.000000,1.000000,1.000000,1.000000,1.000000
53,0.745763,0.915254,0.887006,0.745763,0.706215
54,0.661017,0.717514,0.717514,0.736347,0.731638
56,0.830508,0.830508,0.830508,0.689266,0.661017


In [9]:
scores.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
GPT 4.1,65.0,0.87,0.09,0.66,0.83,0.89,0.94,1.0
GPT 4.1 FT,65.0,0.89,0.09,0.63,0.83,0.92,0.94,1.0
GPT 5.4,65.0,0.88,0.07,0.70,0.83,0.89,0.94,1.0
Mistral Large 3,65.0,0.85,0.09,0.58,0.79,0.86,0.92,1.0
Opus 4.6,65.0,0.87,0.09,0.66,0.80,0.90,0.94,1.0


In [39]:
colori = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#F0E442", "#56B4E9"]
sns.set_palette(colori)

# Pairwise Bootrstap Test with Holm-Bonferroni correction
Per ogni coppia di modelli:

- calcola le differenze appaiate
- Bootstrap della media delle differenze (10.000 campionamenti)
- Intervallo di confidenza al 95% (percentile)
- p-value bootstrap (two-sided)

Poi applica Bonferroni su t-value

In [16]:
# Seed
np.random.seed(13042026)

In [ ]:
# Load
df = scores
models = df.columns.to_list()
n = len(df)
print(f'Records: {n} | Models: {len(models)}')

Records: 65 | Models: 5


In [47]:
# Bootsrap differences
N_BOOT = 10_000
ALPHA = 0.05
results = []

for m_a, m_b in combinations(models, 2):
    print(f'{m_a} vs {m_b}')
    diff = df[m_a].values - df[m_b].values
    observed_mean = diff.mean()
    
    # Bootsrap
    boot_means = np.array([
        diff[np.random.randint(0, n, size=n)].mean()
        for _ in range(N_BOOT)
    ])
    ci_lo, ci_hi = np.percentile(boot_means, [2.5, 97.5])
    
    diff_centered = diff - diff.mean()  # Centrato su 0
    boot_h0 = np.array([
        diff_centered[np.random.randint(0, n, size=n)].mean()
        for _ in range(N_BOOT)
    ])
    p_value = (np.abs(boot_h0) >= np.abs(observed_mean)).mean()
    
    results.append({
        "Model A": m_a,
        "Model B": m_b,
        "Diff media": observed_mean,
        'CI 95% low': ci_lo,
        'CI 95% high': ci_hi,
        'p-value': p_value
    })
    
res_df = pd.DataFrame(results)

# Holm-Bonferroni
k = len(res_df)
sorted_idx = res_df['p-value'].argsort().values
holm_sig = [False] * k

adjusted_alpha_list = [np.nan] * k

for rank, idx in enumerate(sorted_idx):
    adjusted_alpha = ALPHA / (k - rank)
    adjusted_alpha_list[idx] = adjusted_alpha
    if res_df.loc[idx, 'p-value'] <= adjusted_alpha:
        holm_sig[idx] = True
    else:
        break  # Non significativo
    
res_df[f'Adj. alpha'] = adjusted_alpha_list    
res_df['Holm-Bonf. sig.'] = holm_sig
display(res_df)
print(f'Alpha nominale: {ALPHA} | Confronti: {k}')
print(f'Holm-Bonf. sig = True -> differenza significativa dopo correzione')

GPT 4.1 vs GPT 4.1 FT
GPT 4.1 vs GPT 5.4
GPT 4.1 vs Mistral Large 3
GPT 4.1 vs Opus 4.6
GPT 4.1 FT vs GPT 5.4
GPT 4.1 FT vs Mistral Large 3
GPT 4.1 FT vs Opus 4.6
GPT 5.4 vs Mistral Large 3
GPT 5.4 vs Opus 4.6
Mistral Large 3 vs Opus 4.6


,Model A,Model B,Diff media,CI 95% low,CI 95% high,p-value,Adj. alpha,Holm-Bonf. sig.
0,GPT 4.1,GPT 4.1 FT,-0.023208,-0.039046,-0.008121,0.0030,0.005556,True
1,GPT 4.1,GPT 5.4,-0.011130,-0.030132,0.006620,0.2288,NaN,False
2,GPT 4.1,Mistral Large 3,0.020835,0.003035,0.038193,0.0189,0.008333,False
3,GPT 4.1,Opus 4.6,-0.006703,-0.020958,0.007606,0.3571,NaN,False
4,GPT 4.1 FT,GPT 5.4,0.012078,-0.002473,0.026399,0.1016,NaN,False
5,GPT 4.1 FT,Mistral Large 3,0.044044,0.020324,0.067024,0.0005,0.005000,True
6,GPT 4.1 FT,Opus 4.6,0.016505,-0.001444,0.033743,0.0648,NaN,False
7,GPT 5.4,Mistral Large 3,0.031966,0.010429,0.054355,0.0057,0.006250,True
8,GPT 5.4,Opus 4.6,0.004427,-0.014301,0.023829,0.6567,NaN,False
9,Mistral Large 3,Opus 4.6,-0.027538,-0.047237,-0.008128,0.0064,0.007143,True


Alpha nominale: 0.05 | Confronti: 10
Holm-Bonf. sig = True -> differenza significativa dopo correzione
